# 03. Подготовка данных

**Цель:** подготовить признаки и разбиение на train/test без утечки информации.

Порядок работы:

1. Привести `TotalCharges` к числу и обработать пустые значения.
2. Исключить идентификатор `customerID`.
3. Закодировать целевую переменную.
4. Разделить данные на train/test со стратификацией по `Churn`.
5. Собрать `ColumnTransformer` и обучить его только на тренировочной выборке.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_csv(RAW_DATA_PATH)
customer_ids = raw_df["customerID"]

df = raw_df.drop(columns="customerID")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)  # у клиентов с tenure=0 платежей ещё не было
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

print("Пропуски после подготовки:", int(df.isna().sum().sum()))
print("Совпадающие строки без customerID:", int(df.duplicated().sum()))

df.head()

Пропуски после подготовки: 0
Совпадающие строки без customerID: 22


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


После удаления `customerID` **22 строки** полностью совпадают по значениям признаков. Это разные клиенты с одинаковыми характеристиками: исходный `customerID` у них уникален. Удалять такие строки не буду, иначе можно потерять реальных клиентов и исказить распределение.

## 1. Признаки и целевая переменная

In [2]:
TARGET = "Churn"

numeric_features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [
    column for column in df.columns if column not in numeric_features + [TARGET]
]

print("Числовые признаки:", numeric_features)
print("Категориальные признаки:", categorical_features)
print("Доля оттока:", round(df[TARGET].mean(), 4))

Числовые признаки: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Категориальные признаки: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Доля оттока: 0.2654


## 2. Разбиение на train/test

Использую стратификацию, чтобы доля ушедших клиентов была одинаковой в обеих выборках. Тестовая выборка останется нетронутой до финальной оценки в ноутбуке 04.

In [3]:
X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Train: {X_train.shape[0]} строк, test: {X_test.shape[0]} строк")
print("Доля оттока в train:", round(y_train.mean(), 4))
print("Доля оттока в test:", round(y_test.mean(), 4))

Train: 5634 строк, test: 1409 строк
Доля оттока в train: 0.2654
Доля оттока в test: 0.2654


## 3. Препроцессинг

Числовые признаки заполняю медианой и масштабирую. Категориальные — заполняю наиболее частым значением и кодирую через One-Hot Encoding.

`ColumnTransformer` обучается только на `X_train`. В ноутбуке 04 этот же препроцессинг будет встроен в `Pipeline`, чтобы внутри кросс-валидации он обучался заново на каждом фолде.

In [4]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="if_binary",
                sparse_output=False,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print("Размерность после кодирования:", X_train_processed.shape)
pd.DataFrame(X_train_processed[:5], columns=feature_names).head()

Размерность после кодирования: (5634, 40)


,num__SeniorCitizen,num__tenure,num__MonthlyCharges,num__TotalCharges,cat__gender_Male,cat__Partner_Yes,cat__Dependents_Yes,cat__PhoneService_Yes,cat__MultipleLines_No,cat__MultipleLines_No phone service,...,cat__StreamingMovies_No internet service,cat__StreamingMovies_Yes,cat__Contract_Month-to-month,cat__Contract_One year,cat__Contract_Two year,cat__PaperlessBilling_Yes,cat__PaymentMethod_Bank transfer (automatic),cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check
0,-0.441773,0.102371,-0.521976,-0.262257,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,-0.441773,-0.711743,0.337478,-0.503635,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.441773,-0.793155,-0.809013,-0.749883,1.0,1.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,-0.441773,-0.263980,0.284384,-0.172722,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
4,-0.441773,-1.281624,-0.676279,-0.989374,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


## 4. Сохранение подготовленных данных

In [5]:
X_train.to_csv(PROCESSED_DIR / "X_train.csv", index=False)
X_test.to_csv(PROCESSED_DIR / "X_test.csv", index=False)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False)

test_customer_ids = customer_ids.loc[X_test.index].reset_index(drop=True)
test_customer_ids.to_csv(PROCESSED_DIR / "test_customer_ids.csv", index=False)

for path in sorted(PROCESSED_DIR.glob("*.csv")):
    print(f"{path.name}: {path.stat().st_size:,} байт")

test_customer_ids.csv: 16,920 байт
X_test.csv: 176,115 байт
X_train.csv: 702,596 байт
y_test.csv: 4,234 байт
y_train.csv: 16,909 байт


## 5. Выводы

- `TotalCharges` приведён к числу; 11 пустых значений заполнены нулём, потому что относятся к клиентам с нулевым стажем.
- `customerID` исключён из признаков и сохранён отдельно для тестовой выборки — он понадобится только для бизнес-интерпретации.
- Получено **5 634 строки в train** и **1 409 строк в test**; доля оттока в обеих выборках — **26,54%**.
- После One-Hot Encoding получилось **40 признаков**.
- Все обучаемые преобразования выполнены только на тренировочной выборке. Это предотвращает утечку данных из теста.
- Обработанные файлы находятся в `data/processed/` и не публикуются в Git: их можно воспроизвести запуском ноутбуков.